# Literacy Trend Analysis

## Project Overview
Analyze literacy-related assessment data over time and geography to identify progress, stagnation, and disparities in reading and writing proficiency.

## Learning Objectives
- Track literacy rates and assessment scores over time
- Compare literacy outcomes across regions and demographics
- Identify stagnation and improvement trends
- Quantify literacy gaps between population segments

## Problem Statement
Education ministries need to track literacy progress to evaluate policy effectiveness, identify underserved populations, and allocate remediation resources.

## Why This Project Matters
Literacy is foundational to economic participation and social mobility. Tracking trends reveals whether interventions are working and where gaps persist.

## Dataset Overview
Synthetic country-region-year dataset: 8 regions × 10 years × 4 demographic groups, with literacy rate and assessment score metrics.

## Dataset Source and License Notes
- Synthetic data modeled on UNESCO-style literacy indicators
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
regions = ['North', 'South', 'East', 'West', 'Central', 'Coastal', 'Mountain', 'Island']
years = list(range(2014, 2024))
groups = ['Male', 'Female', 'Urban', 'Rural']

rows = []
for region in regions:
    base_rate = np.random.uniform(65, 90)
    base_score = np.random.uniform(250, 350)
    for year in years:
        year_idx = year - 2014
        for group in groups:
            group_offset = {'Male': 2, 'Female': -1, 'Urban': 5, 'Rural': -6}[group]
            trend = year_idx * np.random.uniform(0.3, 1.2)  # improvement per year
            rate = np.clip(base_rate + group_offset + trend + np.random.normal(0, 1.5), 40, 99.5)
            score = np.clip(base_score + group_offset * 5 + trend * 3 + np.random.normal(0, 8), 150, 500)
            pop_tested = np.random.randint(500, 5000)
            rows.append({
                'Region': region, 'Year': year, 'Group': group,
                'LiteracyRate': round(rate, 1), 'AvgScore': round(score, 1),
                'PopTested': pop_tested
            })

df = pd.DataFrame(rows)
print(f'Dataset: {df.shape}')
print(f'Years: {df["Year"].min()} — {df["Year"].max()}')
print(f'Regions: {df["Region"].nunique()}, Groups: {df["Group"].nunique()}')
df.head(8)

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Records per year: {df.groupby("Year").size().unique()}')
print(f'Literacy rate range: {df["LiteracyRate"].min()} — {df["LiteracyRate"].max()}')

## National Literacy Trend

In [ ]:
national = df.groupby('Year').agg(
    AvgLiteracy=('LiteracyRate', 'mean'),
    AvgScore=('AvgScore', 'mean')
).round(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
national['AvgLiteracy'].plot(ax=axes[0], marker='o', color='steelblue')
axes[0].set_title('National Average Literacy Rate Over Time')
axes[0].set_ylabel('Literacy Rate (%)')
national['AvgScore'].plot(ax=axes[1], marker='o', color='coral')
axes[1].set_title('National Average Assessment Score Over Time')
axes[1].set_ylabel('Score')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'national_trend.png'), dpi=100, bbox_inches='tight')
plt.show()

## Regional Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
region_latest = df[df['Year'] == 2023].groupby('Region')['LiteracyRate'].mean().sort_values()
region_latest.plot.barh(ax=axes[0], edgecolor='black', color='teal')
axes[0].set_title('Literacy Rate by Region (2023)')
axes[0].set_xlabel('Literacy Rate (%)')

# Trend by region
for region in regions:
    sub = df[df['Region'] == region].groupby('Year')['LiteracyRate'].mean()
    axes[1].plot(sub.index, sub.values, marker='.', label=region, alpha=0.7)
axes[1].set_title('Literacy Trend by Region')
axes[1].set_ylabel('Literacy Rate (%)')
axes[1].legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'regional_comparison.png'), dpi=100, bbox_inches='tight')
plt.show()

## Demographic Group Gaps

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for group in groups:
    sub = df[df['Group'] == group].groupby('Year')['LiteracyRate'].mean()
    axes[0].plot(sub.index, sub.values, marker='o', label=group)
axes[0].set_title('Literacy Rate by Demographic Group')
axes[0].set_ylabel('Literacy Rate (%)')
axes[0].legend()

# Gap: Urban - Rural
urban = df[df['Group'] == 'Urban'].groupby('Year')['LiteracyRate'].mean()
rural = df[df['Group'] == 'Rural'].groupby('Year')['LiteracyRate'].mean()
gap = urban - rural
gap.plot(ax=axes[1], marker='o', color='red')
axes[1].set_title('Urban-Rural Literacy Gap Over Time')
axes[1].set_ylabel('Gap (percentage points)')
axes[1].axhline(0, color='gray', linestyle='--')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'demographic_gaps.png'), dpi=100, bbox_inches='tight')
plt.show()
print(f'Urban-Rural gap 2014: {gap.iloc[0]:.1f} pp')
print(f'Urban-Rural gap 2023: {gap.iloc[-1]:.1f} pp')

## Improvement Rate Analysis

In [ ]:
# Calculate improvement per region (2014 vs 2023)
first_year = df[df['Year'] == 2014].groupby('Region')['LiteracyRate'].mean()
last_year = df[df['Year'] == 2023].groupby('Region')['LiteracyRate'].mean()
improvement = (last_year - first_year).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['green' if v > 5 else 'orange' if v > 2 else 'red' for v in improvement]
improvement.plot.barh(ax=ax, color=colors, edgecolor='black')
ax.set_title('Literacy Rate Improvement by Region (2014→2023)')
ax.set_xlabel('Change (percentage points)')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'improvement_rate.png'), dpi=100, bbox_inches='tight')
plt.show()
print('Improvement by region (pp):')
print(improvement.round(1))

## Score Distribution by Year

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=df, x='Year', y='AvgScore', ax=ax)
ax.set_title('Assessment Score Distribution by Year')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'score_by_year.png'), dpi=100, bbox_inches='tight')
plt.show()

## Interpretation and Business Insight
- **National literacy is improving** but at a modest pace — sustained effort is required
- **Regional disparities persist** — some regions lag by 10+ percentage points
- **Urban-rural gap** remains significant even as both groups improve — rural areas need targeted investment
- **Gender gap** is narrower than the urban-rural gap in this data, suggesting geographic access is the bigger barrier
- **Improvement rates vary** — regions with the lowest baseline often show the fastest gains (catch-up effect)
- Assessment scores and literacy rates track together, validating the measurement approach

## Limitations
- Synthetic data with simplified trend model
- No individual-level data
- No dropout or out-of-school population tracking
- No intervention-level data to attribute improvements
- Population tested may not be representative

## How to Improve This Project
- Add age-disaggregated data (youth vs adult literacy)
- Include out-of-school population estimates
- Link to specific policy interventions and their timing
- Add multilingual literacy tracking
- Build forecasting models for future literacy rates

## Production Considerations
- National literacy monitoring dashboards
- Automated disparity alerts for policy teams
- Regional benchmarking reports
- Integration with school enrollment and funding data

## Common Mistakes
- Treating literacy as binary (literate/illiterate) when it's a spectrum
- Comparing regions without adjusting for demographics
- Ignoring measurement changes over time (test difficulty, methodology shifts)
- Celebrating national averages that mask severe regional disparities

## Mini Challenge / Exercises
1. Which region has the smallest improvement over the 10-year period?
2. Has the gender gap (Male vs Female literacy) narrowed or widened?
3. Calculate the year when average national literacy crossed 80%.
4. Which region-group combination has the lowest literacy rate in 2023?

## Final Summary / Key Takeaways
- Literacy is improving nationally but unevenly across regions and demographics
- Urban-rural and regional gaps are the dominant disparities
- Trend analysis over 10 years reveals both progress and persistent challenges
- Data-driven literacy monitoring is essential for equitable education policy
- Closing gaps requires targeted interventions in underperforming regions and rural areas